# SentinelDesk — Queue Classifier (Model 2)

Fine-tunes the queue router with the same recipe as the type and priority checkpoints:

* **Data:** `Tobi-Bueck/customer-support-tickets` at the pinned revision, shuffled with seed 42, split 80/20. Only the historical 80% is used here; the 20% holdout is never touched.
* **Labels:** all 52 queues in the historical split (10 helpdesk queues plus 42 topic queues).
* **Base model:** `google-bert/bert-base-multilingual-cased` with a fresh classification head.
* **Input:** subject and body as a sentence pair (blank subjects allowed), truncated to 128 tokens.
* **Splits inside the 80%:** 80% fit / 10% validation / 10% calibration, stratified by queue.
* **Confidence:** a temperature fitted on the calibration rows, saved in `triage_metadata.json`.

Output: `sentineldesk-queue.zip`, ready to drop into `models/queue/`.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
# Colab only: install the same library versions the app uses.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "transformers>=5.13.1,<6" "datasets>=5.0.0" accelerate scikit-learn

## 1. Configuration

In [ ]:
import os

BASE_MODEL = os.environ.get("QUEUE_BASE_MODEL", "google-bert/bert-base-multilingual-cased")
DATASET = "Tobi-Bueck/customer-support-tickets"
DATASET_REVISION = "ddf1c81a5475992c4fa6752bf1e8b4e31f07bbeb"  # same revision as type/priority
SEED = 42
MAX_TOKENS = 128
EPOCHS = float(os.environ.get("QUEUE_EPOCHS", 1))
LEARNING_RATE = 2e-5
BATCH_SIZE = int(os.environ.get("QUEUE_BATCH_SIZE", 32))  # 32 fits a T4; lower it on smaller GPUs
OUTPUT_DIR = os.environ.get("QUEUE_OUTPUT_DIR", "sentineldesk-queue")
ZIP_PATH = OUTPUT_DIR.rstrip("/") + ".zip"

# Set QUEUE_SAMPLE_ROWS to a small number for a quick smoke test; 0 trains on everything.
SAMPLE_ROWS = int(os.environ.get("QUEUE_SAMPLE_ROWS", 0))

## 2. Load the pinned dataset and rebuild the historical 80% split

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset

raw = load_dataset(DATASET, revision=DATASET_REVISION, token=False)["train"].to_pandas()
# Identical to data_loader.py: shuffle with seed 42, then the first 80% is the historical KB.
raw = raw.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
raw["ticket_id"] = [f"TKT-{i:05d}" for i in range(len(raw))]
historical = raw.iloc[: int(len(raw) * 0.8)].copy()
print(f"Total rows: {len(raw):,}   historical (80%): {len(historical):,}   holdout (unused): {len(raw) - len(historical):,}")

df = historical.dropna(subset=["queue", "body"]).copy()
df["subject"] = df["subject"].fillna("").astype(str)
df["body"] = df["body"].astype(str)
if SAMPLE_ROWS:
    df = df.sample(n=min(SAMPLE_ROWS, len(df)), random_state=SEED)
    print(f"SMOKE TEST: using {len(df):,} rows")

queue_counts = df["queue"].value_counts()
print(f"Queues: {len(queue_counts)}")
queue_counts

## 3. Labels and fit / validation / calibration split

In [ ]:
from sklearn.model_selection import train_test_split

labels = sorted(df["queue"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
df["label"] = df["queue"].map(label2id)

def stratify_on(frame):
    # Stratify when every class has at least 2 rows (always true on the full data).
    return frame["label"] if frame["label"].value_counts().min() >= 2 else None

fit_df, rest_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=stratify_on(df))
val_df, cal_df = train_test_split(rest_df, test_size=0.5, random_state=SEED, stratify=stratify_on(rest_df))
print(f"fit: {len(fit_df):,}   validation: {len(val_df):,}   calibration: {len(cal_df):,}")

## 4. Tokenize subject + body as a pair

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["subject", "body", "label"]].reset_index(drop=True))
    return ds.map(
        lambda batch: tokenizer(batch["subject"], batch["body"], truncation=True, max_length=MAX_TOKENS),
        batched=True,
        remove_columns=["subject", "body"],
    )

fit_ds, val_ds, cal_ds = to_dataset(fit_df), to_dataset(val_df), to_dataset(cal_df)

## 5. Fine-tune

In [ ]:
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=list(range(len(labels))), zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

args = TrainingArguments(
    output_dir="queue-training-runs",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=0.1,  # Transformers 5: a float below 1 is a ratio of total steps
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=fit_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()
validation = trainer.evaluate()
validation

## 6. Fit the confidence temperature on the calibration rows

The app divides the logits by this temperature before softmax. Without it, queue confidence is 0 and every ticket goes to human review.

In [ ]:
cal_output = trainer.predict(cal_ds)
cal_logits = torch.tensor(cal_output.predictions, dtype=torch.float32)
cal_labels = torch.tensor(cal_output.label_ids, dtype=torch.long)

log_t = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
nll = torch.nn.CrossEntropyLoss()

def closure():
    optimizer.zero_grad()
    loss = nll(cal_logits / log_t.exp(), cal_labels)
    loss.backward()
    return loss

optimizer.step(closure)
temperature = float(log_t.exp().item())
before = nll(cal_logits, cal_labels).item()
after = nll(cal_logits / temperature, cal_labels).item()
print(f"temperature = {temperature:.4f}   calibration NLL {before:.4f} -> {after:.4f}")
assert np.isfinite(temperature) and temperature > 0

## 7. Save the checkpoint and `triage_metadata.json`

In [ ]:
import json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metadata = {
    "temperature": temperature,
    "max_tokens": MAX_TOKENS,
    "input_format": "paired",
    "validation": {
        "eval_loss": validation["eval_loss"],
        "eval_macro_f1": validation["eval_macro_f1"],
        "eval_accuracy": validation["eval_accuracy"],
    },
    "fit_rows": len(fit_df),
    "validation_rows": len(val_df),
    "calibration_rows": len(cal_df),
    "num_queues": len(labels),
    "base_model": BASE_MODEL,
    "dataset_revision": DATASET_REVISION,
    "seed": SEED,
}
with open(os.path.join(OUTPUT_DIR, "triage_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

## 8. Reload from disk and run a test prediction (as the app does)

In [ ]:
reloaded_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR, local_files_only=True)
reloaded = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR, local_files_only=True).eval()
assert not any(l.startswith("LABEL_") for l in reloaded.config.id2label.values()), "id2label must hold real queue names"

def predict_queue(subject, body):
    inputs = reloaded_tok(subject or "", body, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)
    with torch.inference_mode():
        probs = (reloaded(**inputs).logits[0] / temperature).softmax(-1)
    idx = int(probs.argmax())
    return reloaded.config.id2label[idx], float(probs[idx])

for _, row in val_df.head(5).iterrows():
    pred, conf = predict_queue(row["subject"], row["body"])
    print(f"{conf:5.1%}  predicted: {pred:<40} true: {row['queue']}")

## 9. Package `sentineldesk-queue.zip` (files at the root, like type/priority)

In [ ]:
import zipfile

ARCHIVE_FILES = {
    "config.json", "model.safetensors", "pytorch_model.bin", "tokenizer.json",
    "tokenizer_config.json", "vocab.txt", "special_tokens_map.json",
    "added_tokens.json", "triage_metadata.json",
}
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in sorted(os.listdir(OUTPUT_DIR)):
        if name in ARCHIVE_FILES:
            archive.write(os.path.join(OUTPUT_DIR, name), arcname=name)

with zipfile.ZipFile(ZIP_PATH) as archive:
    print(archive.namelist())
print(f"{ZIP_PATH}: {os.path.getsize(ZIP_PATH) / 1e6:.0f} MB")

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_PATH)

## 10. Add it to the repo

```bash
cp ~/Downloads/sentineldesk-queue.zip models/queue/
git lfs install
git add models/queue/sentineldesk-queue.zip
git commit -m "Add queue classifier checkpoint"
git push
```

`.gitattributes` already sends `models/*/sentineldesk-*.zip` through Git LFS. Restart the Streamlit app and the "Fallback applied for: queue" note disappears.